# SymptoSense: Complete Diagnostic Model Training & SymCat Knowledge Base on Google Colab

This notebook provides the complete, end-to-end training and compilation pipeline for all three diagnostic tiers of **SymptoSense**:

1. **Tier 1 (41 Core Clinical Conditions)**: Calibrated Random Forest model trained on deduplicated clinical symptom-disease associations (`train_clinical.py`).
2. **Tier 2 (801 CDC Conditions)**: Complete SymCat outpatient conditional probability knowledge base parsed from CDC NAMCS/NHAMCS statistics (`parse_symcat_801.py`).
3. **Tier 3 (49 DDXPlus Benchmark Pathologies)**: Calibrated gradient boosted model (XGBoost) trained on the synthetic DDXPlus patient benchmark with realistic input perturbation and selective prediction operating points (`train_pipeline.py`).

> **Important Medical & Data Disclaimers**:
> - **DDXPlus is a Synthetic Dataset**: The patients and symptom combinations in DDXPlus (Tchango et al., Figshare article 20043374) are computationally generated simulations.
> - **SymCat is an Unvalidated Knowledge Base**: The 801-disease SymCat database represents public health outpatient frequency statistics, not clinical validation or certified triage.
> - **Research Benchmark Numbers**: All metrics and evaluation outputs produced by this notebook represent statistical benchmark numbers, **not clinical validation**.
> - **Not Medical Advice**: SymptoSense is a prototype decision-support tool. It does not provide medical diagnosis or replace licensed clinician evaluation.

### Expected Inputs:
- Official DDXPlus release files downloaded via the Figshare API (`release_train_patients`, `release_validate_patients`, `release_test_patients`, `release_conditions.json`, `release_evidences.json`).
- SHA-256 integrity hashes parsed dynamically from `ml-service/data/ddxplus/SOURCE.md`.
- SymCat 801-disease database (`symcat-801-diseases.csv`, `symcat-474-symptoms.csv`) and clinical datasets tracked in the repository.

### Expected Outputs:
- `symptosense_artifacts.zip` containing:
  - `ml-service/artifacts/` (`best_model.joblib`, `label_encoder.joblib`, `shap_background.joblib`, `urgency_model.joblib`, `feature_columns.json`, `symptom_list.json`, `red_flags.json`).
  - `ml-service/artifacts/clinical/` (`symcat_knowledge_base.json`, `clinical_model.joblib`, `clinical_encoder.joblib`, `disease_descriptions.json`, `disease_precautions.json`, `clinical_report.json`).
  - `ml-service/reports/` (`ml_report.json`, `clinical_report.json`, and diagnostic figures).
  - `lib/ml/symptom-list.json` (frontend evidence schema).
  - `manifest.json` (provenance audit with git hash, pinned package versions, timestamp, and CLI flags).

## 1. Environment Check
Inspect the runtime Python version, system architecture, and GPU hardware availability (`nvidia-smi` and PyTorch CUDA check).

In [ ]:
# [1] Environment Check: Python version and GPU availability
import os, sys, shutil, subprocess

print('=== SYSTEM ENVIRONMENT CHECK ===')
print(f'Python Version : {sys.version}')
print(f'Executable     : {sys.executable}')

has_gpu = False
try:
    import torch
    has_gpu = torch.cuda.is_available()
    print(f'PyTorch CUDA Available : {has_gpu}')
    if has_gpu:
        print(f'GPU Device Name        : {torch.cuda.get_device_name(0)}')
        print(f'Device Count           : {torch.cuda.device_count()}')
except ImportError:
    print('PyTorch not installed in base environment.')

if shutil.which('nvidia-smi'):
    print('\n--- nvidia-smi ---')
    !nvidia-smi
else:
    print('\nGPU Status: No nvidia-smi detected (running in CPU mode).')

## 2. Pipeline Configuration
Set runtime parameters: repository URL, branch, training row budget, optional Google Drive persistence, output directory, and hardware device auto-detection.

In [ ]:
# [2] Configuration Variables
from pathlib import Path
import os

REPO_URL = "https://github.com/taksh1507/SymptoSense.git"
BRANCH = "new-master"
TRAIN_ROWS = 150000  # @param {type:"integer"}
BUILD_SYMCAT = True  # @param {type:"boolean"}
USE_DRIVE = True     # @param {type:"boolean"}
USE_GPU = "auto-detect"  # @param ["auto-detect", "true", "false"]

# Resolve GPU auto-detection
if USE_GPU == "auto-detect":
    try:
        import torch
        resolved_use_gpu = torch.cuda.is_available()
    except ImportError:
        resolved_use_gpu = False
else:
    resolved_use_gpu = str(USE_GPU).lower() == "true"

print(f'Configured Repo     : {REPO_URL} (branch: {BRANCH})')
print(f'Training Row Budget : {TRAIN_ROWS}')
print(f'Build SymCat KB     : {BUILD_SYMCAT}')
print(f'USE_DRIVE           : {USE_DRIVE}')
print(f'Resolved USE_GPU    : {resolved_use_gpu}')

# Setup Output Directory and optional Google Drive Mount
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = Path('/content/drive/MyDrive/SymptoSense_Outputs')
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        print(f'✓ Google Drive mounted. Artifacts and logs will save to: {OUTPUT_DIR}')
    except Exception as exc:
        print(f'Google Drive mount failed or not running in Colab ({exc}). Falling back to /content/outputs.')
        OUTPUT_DIR = Path('/content/outputs')
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    OUTPUT_DIR = Path('/content/outputs')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'USE_DRIVE disabled. Output directory set to: {OUTPUT_DIR}')

## 3. Clone or Update Repository
Clone the SymptoSense repository into `/content/SymptoSense` (or pull the latest branch commits if already cloned) and switch the working directory into it.

In [ ]:
# [3] Clone or Update Repository in /content/SymptoSense
import os, subprocess
from pathlib import Path

REPO_DIR = Path("/content/SymptoSense")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    print(f'Found existing repository at {REPO_DIR}. Fetching and checking out {BRANCH}...')
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=REPO_DIR, check=True)
else:
    print(f'Cloning {REPO_URL} (branch: {BRANCH}) into {REPO_DIR}...')
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/SymptoSense
os.chdir(REPO_DIR)
print(f'Working directory set to: {os.getcwd()}')

## 4. Install Pinned Dependencies
Parse `ml-service/requirements.txt`, install exact pinned versions of packages needed for training, and assert that `scikit-learn` and `xgboost` match pins exactly.

In [ ]:
# [4] Install and Assert Pinned Dependencies
import re, sys, subprocess
from pathlib import Path

req_file = Path("ml-service/requirements.txt")
if not req_file.exists():
    raise FileNotFoundError(f'requirements.txt missing at {req_file.resolve()}')

NEEDED_PACKAGES = {
    'numpy', 'scipy', 'pandas', 'scikit-learn', 'xgboost',
    'shap', 'joblib', 'matplotlib', 'pydantic', 'pytest'
}

pins = {}
for line in req_file.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    m = re.match(r'^([a-zA-Z0-9_\-]+)==([a-zA-Z0-9_\.\+]+)', line)
    if m:
        pins[m.group(1).lower()] = m.group(2)

missing_pins = [pkg for pkg in NEEDED_PACKAGES if pkg not in pins]
if missing_pins:
    raise ValueError(f'Missing pinned version for required packages: {missing_pins}')

print('Installing exact pinned versions for training packages...')
for pkg in sorted(NEEDED_PACKAGES):
    spec = f'{pkg}=={pins[pkg]}'
    print(f'Installing {spec}...')
    res = subprocess.run([sys.executable, '-m', 'pip', 'install', spec], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f'ERROR: Failed to install pinned dependency {spec} on Colab Python {sys.version.split()[0]}!\n'
            f'Package: {pkg}, Version: {pins[pkg]}\n'
            f'Pip Error:\n{res.stderr}\n'
            f'Cannot continue: Joblib model artifacts require identical library versions.'
        )

# Re-import and assert exact library versions
import sklearn
import xgboost

print(f'Installed scikit-learn version: {sklearn.__version__} (pinned: {pins["scikit-learn"]})')
print(f'Installed xgboost version     : {xgboost.__version__} (pinned: {pins["xgboost"]})')

assert sklearn.__version__ == pins['scikit-learn'], (
    f'scikit-learn version mismatch: {sklearn.__version__} != {pins["scikit-learn"]}. Stop.'
)
assert xgboost.__version__ == pins['xgboost'], (
    f'xgboost version mismatch: {xgboost.__version__} != {pins["xgboost"]}. Stop.'
)
print('✓ All training library pins successfully verified.')

## 5. Download DDXPlus Dataset via Figshare API
Query the Figshare v2 API (`https://api.figshare.com/v2/articles/20043374`), read the SHA-256 hashes directly from `ml-service/data/ddxplus/SOURCE.md`, download the patient zips and metadata JSONs, extract CSVs, and verify integrity.

In [ ]:
# [5] Download DDXPlus via Figshare API and Verify Hashes from SOURCE.md
import hashlib, json, re, urllib.request, zipfile
from pathlib import Path

DATA_DIR = Path("ml-service/data/ddxplus")
DATA_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_MD = DATA_DIR / "SOURCE.md"

if not SOURCE_MD.exists():
    raise FileNotFoundError(f'SOURCE.md not found at {SOURCE_MD.resolve()}')

# Dynamically parse SHA-256 hashes from SOURCE.md table
EXPECTED_HASHES = {}
for line in SOURCE_MD.read_text(encoding='utf-8').splitlines():
    m = re.search(r'\|\s*`([^`]+)`\s*\|\s*`([a-f0-9]{64})`\s*\|', line)
    if m:
        EXPECTED_HASHES[m.group(1)] = m.group(2)

print(f'Parsed {len(EXPECTED_HASHES)} file hashes from {SOURCE_MD.name}:')
for name, h in EXPECTED_HASHES.items():
    print(f'  {name:<30}: {h[:12]}...')

def sha256_of(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

# Query Figshare v2 API
FIGSHARE_ARTICLE_ID = 20043374
api_url = f'https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}'
print(f'\nFetching metadata from Figshare API: {api_url}...')
req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0 (SymptoSense Colab Training)'})
with urllib.request.urlopen(req, timeout=60) as resp:
    meta = json.loads(resp.read().decode('utf-8'))

api_files = {f['name']: f['download_url'] for f in meta.get('files', [])}
print(f'Discovered {len(api_files)} files in Figshare article.')

# 1. Download conditions and evidences JSON
for json_name in ['release_conditions.json', 'release_evidences.json']:
    target = DATA_DIR / json_name
    exp_sha = EXPECTED_HASHES.get(json_name)
    if not exp_sha:
        raise KeyError(f'{json_name} missing from SOURCE.md hashes!')
    if target.exists() and sha256_of(target) == exp_sha:
        print(f'✓ {json_name} already present and verified.')
        continue
    if json_name not in api_files:
        print('Available files in Figshare response:', sorted(api_files.keys()))
        raise KeyError(f'File {json_name} not found in Figshare API response!')
    print(f'Downloading {json_name}...')
    urllib.request.urlretrieve(api_files[json_name], target)
    actual_sha = sha256_of(target)
    if actual_sha != exp_sha:
        raise ValueError(f'Integrity mismatch for {json_name}: expected {exp_sha}, got {actual_sha}')
    print(f'✓ {json_name} verified (SHA-256 match).')

# 2. Download patient splits and extract under ddxplus_data.SPLITS names
patient_splits = [
    ('release_train_patients.zip', 'release_train_patients'),
    ('release_validate_patients.zip', 'release_validate_patients'),
    ('release_test_patients.zip', 'release_test_patients')
]

for zip_name, csv_name in patient_splits:
    csv_target = DATA_DIR / csv_name
    zip_target = DATA_DIR / zip_name
    exp_csv_sha = EXPECTED_HASHES.get(csv_name)
    exp_zip_sha = EXPECTED_HASHES.get(zip_name)

    if not exp_csv_sha:
        raise KeyError(f'{csv_name} missing from SOURCE.md hashes!')

    # Check if extracted file already matches
    if csv_target.exists() and sha256_of(csv_target) == exp_csv_sha:
        print(f'✓ {csv_name} already extracted and verified.')
        continue

    # Check or download zip archive
    if not (zip_target.exists() and exp_zip_sha and sha256_of(zip_target) == exp_zip_sha):
        if zip_name not in api_files:
            print('Available files in Figshare response:', sorted(api_files.keys()))
            raise KeyError(f'Archive {zip_name} not found in Figshare API response!')
        print(f'Downloading {zip_name}...')
        urllib.request.urlretrieve(api_files[zip_name], zip_target)
        if exp_zip_sha:
            act_zip_sha = sha256_of(zip_target)
            if act_zip_sha != exp_zip_sha:
                raise ValueError(f'Integrity mismatch for {zip_name}: expected {exp_zip_sha}, got {act_zip_sha}')
        print(f'✓ {zip_name} downloaded and verified.')

    print(f'Extracting {zip_name}...')
    with zipfile.ZipFile(zip_target, 'r') as zf:
        zf.extractall(DATA_DIR)
    act_csv_sha = sha256_of(csv_target)
    if act_csv_sha != exp_csv_sha:
        raise ValueError(f'Integrity mismatch for extracted {csv_name}: expected {exp_csv_sha}, got {act_csv_sha}')
    print(f'✓ {csv_name} extracted and verified (SHA-256 match).')

print('\nAll 5 DDXPlus data files successfully verified and ready.')

## 6. Preflight Schema, DDXPlus, and SymCat Data Check
Confirm all 5 DDXPlus files exist in `ml-service/data/ddxplus/`, verify the repository-included SymCat and clinical datasets exist in `ml-service/data/`, import `ml_pipeline.ddxplus_data`, and call `feature_schema()`.

In [ ]:
# [6] Preflight Check: Confirm Data Files & Feature Schema
from pathlib import Path
import sys

ml_dir = Path("ml-service").resolve()
if str(ml_dir) not in sys.path:
    sys.path.insert(0, str(ml_dir))

DATA_DIR = ml_dir / "data" / "ddxplus"
REQUIRED_DDX_FILES = [
    'release_train_patients',
    'release_validate_patients',
    'release_test_patients',
    'release_conditions.json',
    'release_evidences.json'
]

for fname in REQUIRED_DDX_FILES:
    fpath = DATA_DIR / fname
    assert fpath.exists(), f'Preflight failure: Missing DDXPlus file at {fpath}'
print('✓ Confirmed all 5 DDXPlus data files are present.')

# Verify SymCat and Clinical dataset availability
SYMCAT_FILES = [
    ml_dir / 'data' / 'symcat' / 'symcat-801-diseases.csv',
    ml_dir / 'data' / 'symcat' / 'symcat-474-symptoms.csv',
    ml_dir / 'data' / 'dataset.csv'
]
for spath in SYMCAT_FILES:
    assert spath.exists(), f'Preflight failure: Missing SymCat/Clinical dataset at {spath}'
print('✓ Confirmed SymCat 801-disease and Clinical 41 datasets are present.')

from ml_pipeline import ddxplus_data
feature_cols = ddxplus_data.feature_schema()
num_features = len(feature_cols)
print(f'Generated DDXPlus feature schema count: {num_features} features')

EXPECTED_COUNT = 980
if num_features != EXPECTED_COUNT:
    print(f'WARNING: Generated feature count ({num_features}) differs from expected {EXPECTED_COUNT} in benchmark report!')
else:
    print(f'✓ Feature schema dimension matches expected ({EXPECTED_COUNT} features).')

## 7. Detect Training Options and Hardware Device
Run `python ml-service/train_pipeline.py --help` to detect supported command-line flags (`--realistic`, `--aug-copies`, `--no-realistic`), inspect whether the codebase reads `DDX_XGB_DEVICE`, and configure GPU or CPU training.

In [ ]:
# [7] Detect Training Options and Hardware Device
import os, subprocess, sys
from pathlib import Path

# 1. Run train_pipeline.py --help
help_proc = subprocess.run([sys.executable, "ml-service/train_pipeline.py", "--help"], capture_output=True, text=True)
help_text = help_proc.stdout

has_realistic = "--realistic" in help_text
has_aug_copies = "--aug-copies" in help_text
has_no_realistic = "--no-realistic" in help_text

train_flags = []
if has_realistic and has_aug_copies:
    train_flags.extend(["--realistic", "--aug-copies", "1"])
    print('Detected realistic perturbation options: --realistic and --aug-copies 1 added.')
elif has_realistic:
    train_flags.append("--realistic")
    print('Detected realistic flag: --realistic added.')
else:
    print('No realistic flags detected; using default train_pipeline arguments.')

# 2. Check if repo code reads DDX_XGB_DEVICE
reads_xgb_device = False
ml_dir = Path("ml-service")
for py_file in ml_dir.glob("**/*.py"):
    try:
        if "DDX_XGB_DEVICE" in py_file.read_text(encoding="utf-8"):
            reads_xgb_device = True
            print(f'Found DDX_XGB_DEVICE reference in: {py_file.relative_to(ml_dir)}')
            break
    except Exception:
        pass

if resolved_use_gpu and reads_xgb_device:
    os.environ["DDX_XGB_DEVICE"] = "cuda"
    print('Hardware configuration: DDX_XGB_DEVICE=cuda set for XGBoost GPU training.')
else:
    os.environ["DDX_XGB_DEVICE"] = "cpu"
    if not resolved_use_gpu:
        print('Hardware configuration: Training on CPU (USE_GPU is false or no CUDA GPU available).')
    else:
        print('Hardware configuration: Training on CPU (repo code does not read DDX_XGB_DEVICE).')

print(f'Configured training options: {train_flags}')

## 8. Train Multi-Tier Diagnostic Pipeline (SymCat, Clinical, DDXPlus)
Compile the complete SymCat 801 CDC Knowledge Base (`parse_symcat_801.py`), train the 41-condition Calibrated Clinical Classifier (`train_clinical.py`), and execute `python train_pipeline.py --train-rows {TRAIN_ROWS}` from `ml-service/`, streaming logs to `OUTPUT_DIR/training.log`.

In [ ]:
# [8] Train SymCat 801 KB, Clinical 41 Model, and DDXPlus Pipeline
import os, sys, time, subprocess
from pathlib import Path

cwd = Path("ml-service").resolve()
log_path = OUTPUT_DIR / "training.log"

print('='*70)
print('STEP 1: Compiling SymCat 801-Disease CDC Knowledge Base (parse_symcat_801.py)...')
symcat_proc = subprocess.run([sys.executable, "parse_symcat_801.py"], cwd=str(cwd), capture_output=True, text=True)
print(symcat_proc.stdout)
if symcat_proc.returncode != 0:
    print(symcat_proc.stderr)
    raise RuntimeError('SymCat 801 Knowledge Base compilation failed!')
print('✓ SymCat 801 Knowledge Base compiled successfully.')

print('\n' + '='*70)
print('STEP 2: Training Calibrated Clinical 41-Disease Model (train_clinical.py)...')
clin_proc = subprocess.run([sys.executable, "train_clinical.py"], cwd=str(cwd), capture_output=True, text=True)
print(clin_proc.stdout)
if clin_proc.returncode != 0:
    print(clin_proc.stderr)
    raise RuntimeError('Clinical 41-Disease model training failed!')
print('✓ Clinical 41-Disease model trained successfully.')

print('\n' + '='*70)
print('STEP 3: Training DDXPlus Realistic Pipeline (train_pipeline.py)...')
cmd = [sys.executable, "train_pipeline.py", "--train-rows", str(TRAIN_ROWS)] + train_flags
print(f'Command : {" ".join(cmd)}')
print(f'Log file: {log_path}\n' + '-'*70)

start_time = time.perf_counter()

with open(log_path, 'w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
        log_file.write(line)
        log_file.flush()
    process.stdout.close()
    return_code = process.wait()

elapsed = time.perf_counter() - start_time
print('-'*70)
print(f'DDXPlus training exited with code: {return_code}')
print(f'Elapsed execution time: {elapsed:.2f} seconds ({elapsed / 60:.2f} minutes)')

if return_code != 0:
    raise RuntimeError(f'Training pipeline exited with error code {return_code}. Check log at {log_path}')
print(f'✓ All diagnostic tiers successfully trained and compiled. Log saved to {log_path}')

## 9. Display Report Tables and Generated Figures
Load `ml-service/reports/ml_report.json` and `ml-service/reports/clinical_report.json` and build structured evaluation tables dynamically from their fields. Render saved diagnostic figures from `ml-service/reports/figures/`.

In [ ]:
# [9] Display Performance Tables and Visualizations from ml_report.json & clinical_report.json
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 1. Display Clinical 41-Disease Model Report
clin_report_path = Path("ml-service/reports/clinical_report.json")
if clin_report_path.exists():
    clin_report = json.loads(clin_report_path.read_text(encoding='utf-8'))
    print('=== TIER 1: CLINICAL 41-DISEASE MODEL METRICS ===')
    print(f"Model Name : {clin_report.get('model_name')}")
    print(f"Unique Rows: {clin_report.get('dataset', {}).get('unique_rows')} (deduped from {clin_report.get('dataset', {}).get('raw_rows')})")
    display(pd.DataFrame([clin_report.get('holdout_test_metrics', {})]))

# 2. Display DDXPlus Benchmark Model Report
report_file = Path("ml-service/reports/ml_report.json")
if not report_file.exists():
    raise FileNotFoundError(f'Report file missing at {report_file.resolve()}')

report = json.loads(report_file.read_text(encoding='utf-8'))
print('\n=== TIER 3: DDXPLUS BENCHMARK MODEL REPORT ===')
print(f"Model Version : {report.get('model_version')}")
print(f"Generated At  : {report.get('generated_at')}")
print(f"Winning Model : {report.get('training', {}).get('winner')}")

comp_data = report.get('training', {}).get('comparison', [])
if comp_data:
    print('\n--- Model Comparison (CV) ---')
    display(pd.DataFrame(comp_data))

calib = report.get('calibration', {})
if calib:
    print('\n--- Calibration Decision ---')
    display(pd.DataFrame([{
        'Selected Method': calib.get('method'),
        'Brier Before': calib.get('brier_before'),
        'Brier After': calib.get('brier_after'),
        'ECE Before': calib.get('ece_before'),
        'ECE After': calib.get('ece_after')
    }]))

metrics = report.get('evaluation', {}).get('metrics', {})
if metrics:
    print('\n--- Test Metrics ---')
    display(pd.DataFrame([metrics]))

robustness = report.get('evaluation', {}).get('robustness', [])
if robustness:
    print('\n--- Evidence Dropping Robustness ---')
    display(pd.DataFrame(robustness))

urgency = report.get('urgency', {})
if urgency:
    print('\n--- Urgency Prediction Summary ---')
    display(pd.DataFrame([{
        'Severity Accuracy': urgency.get('accuracy_against_derived_labels'),
        'High Urgency Recall': urgency.get('high_recall'),
        'Rule Agreement': urgency.get('rule_agreement'),
        'Rule Agreement After Red Flags': urgency.get('rule_agreement_after_red_flags'),
        'Rule Context': urgency.get('rule_context')
    }]))

for sec in ['realistic_input', 'selective_prediction', 'data_hygiene']:
    val = report.get(sec)
    if val:
        print(f'\n--- {sec.upper().replace("_", " ")} ---')
        if isinstance(val, dict):
            flat = {k: v for k, v in val.items() if not isinstance(v, (dict, list))}
            display(pd.DataFrame([flat]))
        elif isinstance(val, list):
            display(pd.DataFrame(val))

# 3. Render figures inline
fig_dir = Path("ml-service/reports/figures")
if fig_dir.exists():
    for fig_path in sorted(fig_dir.glob("*.png")):
        print(f'\nDiagnostic Figure: {fig_path.name}')
        plt.figure(figsize=(9, 5))
        img = mpimg.imread(str(fig_path))
        plt.imshow(img)
        plt.axis('off')
        plt.title(fig_path.stem.replace('_', ' ').title(), fontsize=12)
        plt.show()

## 10. Sanity Check, Multi-Tier Test Inference, and Unit Test Suite
Verify `best_model.joblib` with `to_sparse` / `predict_proba`, verify SymCat 801 KB (`symcat_knowledge_base.json`) and clinical model (`clinical_model.joblib`), test multi-tier inference via `disease_inference.py`, and run `pytest ml-service/tests -q`.

In [ ]:
# [10] Sanity Check: Model Predictions, SymCat Inference & Pytest Verification
import json, joblib, subprocess, sys
import numpy as np
from pathlib import Path
from ml_pipeline import model_input

# 1. Verify DDXPlus Model and Probability Normalization
art_dir = Path("ml-service/artifacts")
best_model = joblib.load(art_dir / "best_model.joblib")
label_encoder = joblib.load(art_dir / "label_encoder.joblib")
feature_cols = json.loads((art_dir / "feature_columns.json").read_text(encoding='utf-8'))

print(f'Loaded DDXPlus Model  : {type(best_model).__name__}')
print(f'Label Encoder Classes : {len(label_encoder.classes_)}')
print(f'Feature Schema Width  : {len(feature_cols)}')

rng = np.random.default_rng(42)
test_dense = np.zeros((20, len(feature_cols)), dtype=np.float32)
test_dense[:, 0] = rng.uniform(18, 65, size=20)  # age
test_dense[:, 5] = 1.0  # sex_f
for i in range(20):
    active_idx = rng.choice(np.arange(8, len(feature_cols)), size=rng.integers(3, 7), replace=False)
    test_dense[i, active_idx] = 1.0

sparse_inputs = model_input.to_sparse(test_dense)
probabilities = model_input.predict_proba(best_model, sparse_inputs)

assert np.all(np.isfinite(probabilities)), 'Sanity check failure: non-finite probabilities found!'
row_sums = np.sum(probabilities, axis=1)
assert np.allclose(row_sums, 1.0, atol=1e-4), f'Sanity check failure: probabilities do not sum to 1: {row_sums}'
assert probabilities.shape[1] == len(label_encoder.classes_), (
    f'Class count mismatch: model produces {probabilities.shape[1]} outputs, encoder has {len(label_encoder.classes_)} classes'
)
print('✓ DDXPlus model probability sanity checks passed: finite, normalized sums, matching class count.')

# 2. Verify SymCat 801 Knowledge Base & Clinical Model
clin_art = art_dir / "clinical"
symcat_kb_path = clin_art / "symcat_knowledge_base.json"
assert symcat_kb_path.exists(), f'Missing SymCat Knowledge Base at {symcat_kb_path}'
symcat_kb = json.loads(symcat_kb_path.read_text(encoding='utf-8'))
print(f"✓ SymCat 801 KB Verified: {symcat_kb.get('total_conditions')} conditions, {symcat_kb.get('total_symptoms')} symptoms.")

# 3. Test Full Multi-Tier Routing via disease_inference.py
from disease_inference import predict_disease, DiseaseInput
sample_res = predict_disease(DiseaseInput(symptoms=['fever', 'cough'], age=32.0, sex='Male'))
print('\n=== INFERENCE TEST (disease_inference.py) ===')
print(f"Model Version   : {sample_res.get('model_version')}")
print(f"Top 1 Condition : {sample_res['top3'][0]['disease']} ({sample_res['top3'][0]['probability']*100:.1f}%)")
print(f"Urgency Level   : {sample_res.get('urgency')}")
print(f"Operating Point : {sample_res.get('operating_point')}")
print(f"Context Note    : {sample_res.get('context_note')}")

# 4. Run Pytest Test Suite
print('\nRunning pytest test suite: pytest ml-service/tests -q...')
test_proc = subprocess.run([sys.executable, '-m', 'pytest', 'ml-service/tests', '-q'], capture_output=True, text=True)
print(test_proc.stdout)
if test_proc.stderr:
    print('Pytest STDERR:\n' + test_proc.stderr)
print(f'Pytest exit code: {test_proc.returncode}')

## 11. Package All Artifacts and Download
Package all trained models across all three diagnostic tiers (`ml-service/artifacts/` including `clinical/` and `symcat_knowledge_base.json`), `ml-service/reports/`, `lib/ml/symptom-list.json`, and `manifest.json` into `symptosense_artifacts.zip`. Save to `OUTPUT_DIR`, offer Colab download, and display unpacking commands.

In [ ]:
# [11] Package All Artifacts into symptosense_artifacts.zip with manifest.json
import json, os, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path

# 1. Create manifest.json with environment and run provenance
git_hash = 'unknown'
try:
    git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    pass

import sklearn, xgboost, numpy, scipy, pandas, shap, joblib, matplotlib, pydantic

manifest_data = {
    'git_commit_hash': git_hash,
    'python_version': sys.version,
    'library_versions': {
        'scikit-learn': sklearn.__version__,
        'xgboost': xgboost.__version__,
        'numpy': numpy.__version__,
        'scipy': scipy.__version__,
        'pandas': pandas.__version__,
        'shap': shap.__version__,
        'joblib': joblib.__version__,
        'matplotlib': matplotlib.__version__,
        'pydantic': pydantic.__version__
    },
    'run_timestamp': datetime.now(timezone.utc).isoformat(),
    'train_rows': TRAIN_ROWS,
    'flags_used': train_flags,
    'tiers_packaged': ['Tier 1 (Clinical 41)', 'Tier 2 (SymCat 801 CDC KB)', 'Tier 3 (DDXPlus 49)']
}

manifest_path = Path('manifest.json')
manifest_path.write_text(json.dumps(manifest_data, indent=2), encoding='utf-8')
print('✓ Generated manifest.json:')
print(json.dumps(manifest_data, indent=2))

# 2. Build symptosense_artifacts.zip
zip_target = OUTPUT_DIR / 'symptosense_artifacts.zip'
print(f'\nPackaging all multi-tier artifacts into: {zip_target}...')

with zipfile.ZipFile(zip_target, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    # Add all artifacts (including clinical/ and symcat_knowledge_base.json)
    for p in Path('ml-service/artifacts').rglob('*'):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())
    # Add all reports (ml_report.json, clinical_report.json, and figures)
    for p in Path('ml-service/reports').rglob('*'):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())
    # Add lib/ml/symptom-list.json
    sym_json = Path('lib/ml/symptom-list.json')
    if sym_json.exists():
        zf.write(sym_json, arcname=sym_json.as_posix())
    # Add manifest.json
    zf.write(manifest_path, arcname='manifest.json')

mb_size = zip_target.stat().st_size / (1024 * 1024)
print(f'✓ Successfully packaged {zip_target.name} ({mb_size:.2f} MB).')

# 3. Trigger direct browser download in Colab
try:
    from google.colab import files
    print('Triggering browser download...')
    files.download(str(zip_target))
except Exception as exc:
    print(f'Notice: Browser download not available ({exc}). Archive saved at {zip_target}')

# 4. Print unpacking commands
print('\n' + '='*70)
print('COMMANDS TO UNPACK ALL ARTIFACTS INTO YOUR LOCAL CLONE:')
print('='*70)
print('# 1. Move symptosense_artifacts.zip to your local repo root')
print('# 2. Run one of these commands from the repo root directory:')
print('     python -m zipfile -e symptosense_artifacts.zip .')
print('     # or on Linux/macOS:   unzip -o symptosense_artifacts.zip')
print('     # or on PowerShell:    Expand-Archive -Path symptosense_artifacts.zip -DestinationPath . -Force')
print('# 3. Check modified files with git status')
print('='*70)